# 04 · PPO (Proximal Policy Optimization) 原理与实战

> 前置:必须先跑完 [`02`](02_奖励模型RewardModel原理与实战.ipynb),它会在 `../outputs/reward-model-merged` 生成奖励模型;并跑通 [`00`](00_环境准备与GPU检测.ipynb)。

PPO 是**经典 RLHF** 的算法内核(ChatGPT 早期就是用它对齐的)。相比 DPO,它更「原汁原味」地体现了强化学习:
**策略在线生成 → 奖励模型打分 → 带着两把安全锁(clip + KL)更新策略**。

> ⚠️ **重要预期管理**:PPO 是本仓库**最复杂、最吃显存、也最挑 trl 版本**的一章。
> 它要同时装下 4 个模型。本章代码基于 **trl 0.15+** 的新版 `PPOTrainer` API 编写。
> 目的在于**跑通闭环、看懂每个零件**,而不是追求效果 —— 想要好效果请优先用 DPO(03)。

## 一、PPO 的四模型闭环

```mermaid
flowchart LR
    prompt["prompt<br/>(来自数据集)"] --> policy["策略模型 Policy<br/>(要训练的)"]
    policy -->|"生成回答 (rollout)"| resp["回答 y"]
    resp --> rm["奖励模型 RM<br/>(02 训练的,冻结)"]
    rm -->|"打分 reward"| adv["计算优势 A<br/>(reward - 价值基线)"]
    resp --> value["价值模型 Value<br/>(估计基线,可训练)"]
    value --> adv
    resp --> ref["参考模型 Ref<br/>(冻结的初始策略)"]
    ref -->|"KL 惩罚:别跑太偏"| adv
    adv -->|"PPO 裁剪更新"| policy
```

四个模型各司其职(对应 01 讲过的角色):

- **策略 Policy**:被训练的模型,负责生成。
- **参考 Reference**:冻结的初始策略,提供 KL 惩罚基准(防止「为骗高分而说胡话」)。
- **奖励 Reward**:02 训练好的打分器,给整段回答打分。**冻结**。
- **价值 Value / Critic**:估计「当前状态的期望回报」当基线,用来算优势、降方差。**可训练**。

一步 PPO = 采样一批回答(rollout)→ 打分 → 算优势 → 在同一批数据上做几轮裁剪更新。

## 二、准备采样用的 prompt 数据(优先用 HuggingFace 真实指令)

PPO 是「在线」的:只需要 **prompt**,回答由策略当场生成。
这里**优先从 HuggingFace 拉取**中文指令数据 [`llamafactory/alpaca_gpt4_zh`](https://huggingface.co/datasets/llamafactory/alpaca_gpt4_zh),
取其中的 `instruction` 当采样 prompt(比手写的十几条更丰富);下载失败则回退本地 `data/prompts.jsonl` 里 `task=chat` 的题目。

无论哪种来源,都 tokenize 成 `input_ids`(套上对话模板 + 生成提示),这是新版 `PPOTrainer` 需要的输入列。

In [ ]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # PPO 采样重,用最小尺寸保稳

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_PROMPTS = 64                              # 演示规模:采样用的 prompt 数量

def build_prompt_ids(text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": text}], add_generation_prompt=True)

def load_ppo_prompts():
    """优先用 HuggingFace 中文指令 llamafactory/alpaca_gpt4_zh 的 instruction 当 prompt;
    失败回退本地 data/prompts.jsonl(task=chat)。"""
    try:
        ds = load_dataset("llamafactory/alpaca_gpt4_zh", split="train")
        ds = ds.filter(lambda e: not e.get("input"))       # 只取无额外输入的单轮指令
        ds = ds.shuffle(seed=42).select(range(min(N_PROMPTS, len(ds))))
        ds = ds.map(lambda e: {"input_ids": build_prompt_ids(e["instruction"])},
                    remove_columns=ds.column_names)
        print(f"✅ 已从 HuggingFace 加载 llamafactory/alpaca_gpt4_zh,取 {len(ds)} 条 prompt")
    except Exception as e:
        print(f"⚠️ 从 HF 下载失败({type(e).__name__}),回退本地 data/prompts.jsonl\n   {e}")
        raw = load_dataset("json", data_files="../data/prompts.jsonl", split="train")
        raw = raw.filter(lambda x: x["task"] == "chat")
        ds = raw.map(lambda e: {"input_ids": build_prompt_ids(e["prompt"])},
                     remove_columns=raw.column_names)
    return ds.filter(lambda e: len(e["input_ids"]) <= 128)

ppo_ds = load_ppo_prompts()
print(ppo_ds)
print("示例 input_ids 长度:", len(ppo_ds[0]["input_ids"]))

## 三、加载四个模型

- **策略 / 参考**:`AutoModelForCausalLM`(同一个底座,各加载一份;参考会被冻结)。
- **奖励 / 价值**:`AutoModelForSequenceClassification(num_labels=1)`。直接复用 02 合并保存的奖励模型;价值模型也从它初始化(trl 会自己训练价值头)。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification

RM_DIR = "../outputs/reward-model-merged"   # 来自 02

policy = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
ref_policy = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)

reward_model = AutoModelForSequenceClassification.from_pretrained(
    RM_DIR, num_labels=1, torch_dtype=torch.bfloat16)
value_model = AutoModelForSequenceClassification.from_pretrained(
    RM_DIR, num_labels=1, torch_dtype=torch.bfloat16)

for m in (reward_model,):     # 奖励模型冻结
    for p in m.parameters():
        p.requires_grad_(False)

print("策略 / 参考 / 奖励 / 价值 四个模型已加载。")

## 四、配置 PPOConfig 并训练

关键超参(都往小了设,保证 16GB 跑通):

- `total_episodes`:一共采样多少条回答(控制总训练量)。
- `per_device_train_batch_size` / `num_mini_batches` / `num_ppo_epochs`:一批采样后如何切成小批、重复训几轮。
- `kl_coef`:KL 惩罚系数(那把「别跑太偏」的锁)。
- `response_length`:每条回答最多生成多少 token。
- `missing_eos_penalty`:回答没能正常结束(没生成 eos)时的惩罚,鼓励模型「说完整」。

> 新版 `PPOTrainer` 会把训练指标写进 `trainer.state.log_history`,我们训练完从那里取数据画曲线。

In [ ]:
from trl import PPOConfig, PPOTrainer

ppo_config = PPOConfig(
    output_dir="../outputs/ppo-qwen0.5b",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_mini_batches=1,
    num_ppo_epochs=2,
    total_episodes=48,          # 玩具规模;想要明显效果调大(如 1000+)
    learning_rate=1e-5,
    kl_coef=0.05,
    response_length=64,
    missing_eos_penalty=1.0,
    temperature=0.7,
    num_sample_generations=0,   # 关掉训练中途的示例生成,省时间
    logging_steps=1,
    report_to=[],
)

trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=policy,
    ref_model=ref_policy,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ppo_ds,
)
trainer.train()
print("PPO 训练完成。")

## 五、看 reward 和 KL 的变化

强化学习最该盯的两条曲线:

- **reward 应逐步上升**:策略在学着讨好奖励模型。
- **KL 应被约束在合理范围**:如果 KL 爆炸,说明策略跑偏了(可能在钻奖励模型的空子),要调大 `kl_coef`。

不同 trl 版本记录的指标列名略有差异,下面用「关键字匹配」把 reward 类和 kl 类的列都挑出来画。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.DataFrame(trainer.state.log_history)
print("记录到的指标列:")
print(list(hist.columns))

reward_cols = [c for c in hist.columns if ("reward" in c.lower() or "score" in c.lower())]
kl_cols = [c for c in hist.columns if "kl" in c.lower()]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for c in reward_cols:
    axes[0].plot(hist[c].dropna().reset_index(drop=True), marker="o", label=c)
axes[0].set_title("奖励 (reward) 随训练"); axes[0].set_xlabel("log step"); axes[0].legend(fontsize=8)
for c in kl_cols:
    axes[1].plot(hist[c].dropna().reset_index(drop=True), marker="o", label=c)
axes[1].set_title("KL 随训练"); axes[1].set_xlabel("log step"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 小结

- PPO 是经典 RLHF 的内核:**在线生成 → 奖励模型打分 → clip + KL 两把锁下更新策略**。
- 它要同时维护**四个模型**(策略/参考/奖励/价值),流程复杂、吃显存、对超参和版本都敏感。
- 训练时要盯着 **reward 上升** 和 **KL 不爆炸** 这两条曲线。

**PPO vs DPO 的取舍**:PPO 上限更高、但工程复杂、易抖;DPO 又稳又省、通常是首选。
只有当你需要「在线探索」(奖励来自实时环境/规则、没有现成成对偏好)时,PPO 或下一章的 GRPO 才更合适。

下一站:[`05_GRPO原理与实战`](05_GRPO原理与实战.ipynb) —— PPO 的「瘦身版」,面向可验证任务。